## Layer 1 — Risk Model

### PHASE 1 — Data & Target Finalization


In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.read_csv('/content/loan_data_2007_2014 (1).csv',index_col=0)
columns_to_drop = ['annual_inc_joint','dti_joint','verification_status_joint','open_acc_6m','open_il_6m','open_il_12m','open_il_24m','mths_since_rcnt_il','total_bal_il',
                   'il_util','open_rv_12m','open_rv_24m','max_bal_bc','all_util','inq_fi','total_cu_tl','inq_last_12m','annual_inc_joint','dti_joint','verification_status_joint',
                   'desc','mths_since_last_delinq','mths_since_last_record','next_pymnt_d','mths_since_last_major_derog']
data = data.drop(columns=columns_to_drop)

# Define Target Properly
completed_status = [
    'Fully Paid',
    'Charged Off',
    'Default'
]

data_completed = data[data['loan_status'].isin(completed_status)].copy()

def define_default(status):
    if status in ['Charged Off', 'Default']:
        return 1
    else:
        return 0

data_completed['default_flag'] = data_completed['loan_status'].apply(define_default)

### Data Preprocessing

In [2]:
# drop column title dan emp_title
data_completed = data_completed.drop(columns=['title','emp_title'])

# 1️⃣ Buat missing flag dulu
data_completed['emp_length_missing'] = data_completed['emp_length'].isnull().astype(int)

# 2️⃣ Bersihkan teks
data_completed['emp_length'] = data_completed['emp_length'].str.replace(' years', '', regex=False)
data_completed['emp_length'] = data_completed['emp_length'].str.replace(' year', '', regex=False)
data_completed['emp_length'] = data_completed['emp_length'].str.replace('< 1', '0', regex=False)
data_completed['emp_length'] = data_completed['emp_length'].str.replace('10+', '10', regex=False)

# 3️⃣ Convert ke numeric
data_completed['emp_length'] = pd.to_numeric(data_completed['emp_length'], errors='coerce')

# 4️⃣ Isi missing dengan -1
data_completed['emp_length'] = data_completed['emp_length'].fillna(-1)

data_completed['revol_util'] = data_completed['revol_util'].fillna(0)
data_completed['collections_12_mths_ex_med'] = data_completed['collections_12_mths_ex_med'].fillna(0)

cols_balance = ['tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim']

for col in cols_balance:
    data_completed[col + '_missing_flag'] = data_completed[col].isnull().astype(int)
    data_completed[col] = data_completed[col].fillna(0)


### Data Modeling

In [3]:
# data modeling (Pre-Approval Features)
leakage_columns = [
    'out_prncp',
    'funded_amnt',
    'funded_amnt_inv',
    'out_prncp_inv',
    'total_pymnt',
    'total_pymnt_inv',
    'total_rec_prncp',
    'total_rec_int',
    'total_rec_late_fee',
    'recoveries',
    'collection_recovery_fee',
    'last_pymnt_d',
    'last_pymnt_amnt',
    'last_credit_pull_d'
]

non_model_columns = [
    'id',
    'member_id',
    'url',
    'zip_code',
    'policy_code',
    'loan_status',# karena sudah jadi target

]

drop_columns = leakage_columns + non_model_columns

data_model = data_completed.drop(columns=drop_columns, errors='ignore')

### PHASE 2 — Build PD Model


In [4]:
from sklearn.model_selection import train_test_split

X = data_model.drop(columns=['default_flag'])
y = data_model['default_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# 1️⃣ Identifikasi Numeric vs Categorical
numeric_features = X_train.select_dtypes(include=['int64','float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns


### Data Pipeline

In [5]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.impute import SimpleImputer


numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

calibrated_model = CalibratedClassifierCV(
    estimator=model_pipeline,
    method='isotonic',
    cv=5
)
calibrated_model.fit(X_train, y_train)


# Evaluasi Awal
y_pred_proba_cal = calibrated_model.predict_proba(X_test)[:,1] # PD yang diperlukan artinya (PD_i = probability borrower i akan default)
auc = roc_auc_score(y_test, y_pred_proba_cal)
print("Test AUC:", auc)
print("*******"*10)

y_pred = calibrated_model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(y_pred_proba_cal)

Test AUC: 0.7049261144988581
**********************************************************************
[[36696   252]
 [ 8356   306]]
              precision    recall  f1-score   support

           0       0.81      0.99      0.90     36948
           1       0.55      0.04      0.07      8662

    accuracy                           0.81     45610
   macro avg       0.68      0.51      0.48     45610
weighted avg       0.76      0.81      0.74     45610

[0.24650531 0.14909541 0.31878195 ... 0.08514313 0.19254669 0.13777337]


## Layer 2 — Decision Engine

### Policy Layer

In [6]:
# Simulasi thresholds
thresholds = [0.3, 0.4, 0.5, 0.6]

for t in thresholds:
    y_pred_t = (y_pred_proba_cal >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    approval_rate = (y_pred_t == 0).mean()
    recall_default = tp / (tp + fn)

    print(f"\nThreshold: {t}")
    print("Approval Rate:", round(approval_rate,3))
    print("Default Recall:", round(recall_default,3))


Threshold: 0.3
Approval Rate: 0.818
Default Recall: 0.365

Threshold: 0.4
Approval Rate: 0.937
Default Recall: 0.156

Threshold: 0.5
Approval Rate: 0.988
Default Recall: 0.035

Threshold: 0.6
Approval Rate: 0.999
Default Recall: 0.002


### Simulasi Sederhana

In [7]:
# Simulasi Sederhana
results = []

thresholds = [0.3, 0.4, 0.5, 0.6]

for t in thresholds:
    y_pred_t = (y_pred_proba_cal >= t).astype(int)

    df_temp = pd.DataFrame({
        'actual': y_test,
        'pred_default': y_pred_t,
        'loan_amnt': X_test['loan_amnt'].values,
        'int_rate': X_test['int_rate'].values
    })

    # Approve jika pred_default = 0
    approved = df_temp[df_temp['pred_default'] == 0]

    # Profit dari borrower bagus
    good_loans = approved[approved['actual'] == 0]
    interest_income = (good_loans['loan_amnt'] * good_loans['int_rate'] / 100).sum()

    # Loss dari borrower default
    bad_loans = approved[approved['actual'] == 1]
    loss = (bad_loans['loan_amnt'] * 0.941).sum()

    total_profit = interest_income - loss

    results.append({
        'threshold': t,
        'approved_count': len(approved),
        'interest_income': interest_income,
        'loss': loss,
        'net_profit': total_profit
    })

pd.DataFrame(results)

,threshold,approved_count,interest_income,loss,net_profit
0,0.3,37295,5.037437e+07,6.738433e+07,-1.700995e+07
1,0.4,42724,6.113766e+07,9.489472e+07,-3.375706e+07
2,0.5,45052,6.598775e+07,1.131090e+08,-4.712125e+07
3,0.6,45578,6.702376e+07,1.182189e+08,-5.119515e+07



```
Threshold : 0.4 memiliki net profit tinggi, Loss rendah
```


In [8]:
data_completed['PD'] = calibrated_model.predict_proba(X)[:, 1].round(2)
data_completed['LGD'] = 1 - (data_completed['recoveries']/ data_completed['funded_amnt'])
data_completed['LGD'] = data_completed['LGD'].clip(lower=0, upper=1)
data_completed['EAD'] = data_completed['funded_amnt']
data_completed['Expected_Loss'] = data_completed['PD'] * data_completed['LGD'] * data_completed['EAD']
avg_lgd= data_completed[data_completed['default_flag']==1]['LGD'].mean()

summary = {
    "Metric": [' Avg Probability of Default (PD)', 'Avg Loss Given Default (LGD)', 'Avg Exposure at Default (EAD)','Expected Loss (EL)'],
    "Average Value": [
        data_completed['PD'].mean(),
        data_completed['LGD'].mean(),
        data_completed['EAD'].mean(),
        data_completed['Expected_Loss'].mean()
    ]
}

df_summary = pd.DataFrame(summary)
print("Average LGD Default :", round(avg_lgd,2))
df_summary

Average LGD Default : 0.94


,Metric,Average Value
0,Avg Probability of Default (PD),0.189909
1,Avg Loss Given Default (LGD),0.988682
2,Avg Exposure at Default (EAD),13419.884365
3,Expected Loss (EL),2694.950019


In [9]:
LGD = 0.94

df_profit = pd.DataFrame({
    'PD': y_pred_proba_cal,
    'actual': y_test.values,
    'funded_amnt': data_completed.loc[X_test.index, 'funded_amnt'].values,
    'int_rate': X_test['int_rate'].values
})

# Interest Income
df_profit['interest_income'] = df_profit['funded_amnt'] * df_profit['int_rate'] / 100

# Expected Loss
df_profit['expected_loss'] = df_profit['PD'] * LGD * df_profit['funded_amnt']

# Expected Profit
df_profit['expected_profit'] = (
    (1 - df_profit['PD']) * df_profit['interest_income']
    - df_profit['expected_loss']
)

# Profit Ratio
df_profit['profit_ratio'] = df_profit['expected_profit'] / df_profit['interest_income']

# Decision
df_profit['approve'] = (df_profit['expected_profit'] > 0).astype(int)

# Portfolio Calculation
approved = df_profit[df_profit['approve'] == 1]

good_loans = approved[approved['actual'] == 0]
bad_loans = approved[approved['actual'] == 1]

interest_income = good_loans['interest_income'].sum()
loss = (bad_loans['funded_amnt'] * LGD).sum()

net_profit = interest_income - loss

print("Approved:", len(approved))
print("Net Profit:", net_profit)

Approved: 12785
Net Profit: 3321305.3674999997


**REALISTIC BUSINESS INSIGHT :**
1. Avg PD terlalu tinggi (45%)


```
Expected Loss jadi besar
→ banyak loan terlihat tidak layak
```


2. LGD sangat tinggi (94-98%)

```loss >> interest```



In [10]:
def risk_tier(pd):
    if pd < 0.3:
        return "Low Risk"
    elif pd < 0.6:
        return "Medium Risk"
    else:
        return "High Risk"

df_profit['risk_tier'] = df_profit['PD'].apply(risk_tier)
df_profit['risk_tier'].value_counts() # cek distribusi
df_profit.groupby('risk_tier')['actual'].mean() # cek default rate
df_profit.groupby('risk_tier')['expected_profit'].mean() # cek profit


# Quartile segmentation
df_profit['risk_quartile'] = pd.qcut(
    df_profit['PD'],
    q=4,
    labels=['Q1_Low', 'Q2_MedLow', 'Q3_MedHigh', 'Q4_High']
)

# 1. Membuat rangkuman berdasarkan Risk Tier
tier_summary = df_profit.groupby('risk_tier').agg(
    Total_Nasabah=('risk_tier', 'count'),
    Default_Rate=('actual', 'mean'),
    Avg_Expected_Profit=('expected_profit', 'mean')
).reset_index().rename(columns={'risk_tier': 'Group'})

# 2. Membuat rangkuman berdasarkan Risk Quartile
quartile_summary = df_profit.groupby('risk_quartile').agg(
    Total_Nasabah=('risk_quartile', 'count'),
    Default_Rate=('actual', 'mean'),
    Avg_Expected_Profit=('expected_profit', 'mean')
).reset_index().rename(columns={'risk_quartile': 'Group'})

# 3. Menggabungkan keduanya ke dalam satu tabel summary
tier_summary.insert(0, 'Category', 'Risk Tier')
quartile_summary.insert(0, 'Category', 'Risk Quartile')
final_summary = pd.concat([tier_summary, quartile_summary], ignore_index=True)
final_summary['Default_Rate'] = final_summary['Default_Rate'].map('{:.2%}'.format)
final_summary['Avg_Expected_Profit'] = final_summary['Avg_Expected_Profit'].map('{:,.2f}'.format)

print(final_summary)

        Category        Group  Total_Nasabah Default_Rate Avg_Expected_Profit
0      Risk Tier    High Risk             32       59.38%          -11,031.15
1      Risk Tier     Low Risk          37295       14.76%             -436.70
2      Risk Tier  Medium Risk           8283       37.90%           -4,114.29
3  Risk Quartile       Q1_Low          11413        7.05%              282.67
4  Risk Quartile    Q2_MedLow          11392       12.64%             -166.51
5  Risk Quartile   Q3_MedHigh          11409       21.21%             -992.17
6  Risk Quartile      Q4_High          11396       35.07%           -3,573.89


In [11]:
import joblib

joblib.dump(calibrated_model, "credit_model.pkl")

# ==============================
# SAVE BASE INPUT (FOR STREAMLIT)
# ==============================
# Ambil 1 row dari training sebagai template input
base_input = X_train.iloc[[0]].copy()

# Optional: isi NaN (biar aman)
base_input = base_input.fillna(0)

base_input.to_csv("base_input.csv", index=False)

print("\n✅ Model & base_input saved successfully")


✅ Model & base_input saved successfully
